# Sentinel360: fine-tune the weapon detector (Colab)

Trains YOLO11s on knives and pistols **in people's hands**, plus the things they get mistaken for
(smartphone, wallet, banknote, card) as their own classes, so the model learns the difference.
Data: OD-WeaponDetection / Sohas (Univ. of Granada, CC BY-SA 4.0) and Simuletic CCTV knives (CC BY 4.0).
See `apps/ai/scripts/prepare_weapon_dataset.py`.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Allow Google Drive access when asked: the dataset and training runs are kept there, so a
   disconnect doesn't lose them (re-run all, and the resume cell continues training).
3. Takes ~2–3 h. At the end it downloads `weapon_yolo11s.pt`. Put it in `apps/ai/models/` on the laptop.

In [ ]:
import torch
GPU = torch.cuda.is_available()
if GPU:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('*** NO GPU. Training needs one (CPU would take days). Runtime -> Change runtime type -> T4 GPU. '
          'If T4 is greyed out, Colab\'s free GPU time is used up: try again later (usually a few hours). '
          'The cells below skip training on CPU but can still package a finished model. ***')
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/sentinel360-weapon'
!mkdir -p "$WORK"

In [ ]:
%cd /content
!rm -rf Sentinel360 && git clone --depth 1 --branch model-improvements https://github.com/donnellnaidoo/Sentinel360.git
%cd /content/Sentinel360/apps/ai
# Same Ultralytics as the laptop, so the weights load there.
!pip install -q ultralytics==8.4.100 httpx gdown
# Training runs live on Drive (survive a disconnect).
!mkdir -p "$WORK/runs" && rm -rf runs && ln -s "$WORK/runs" runs

## 1. Build the dataset (~15 min)
Downloads the 4.7 GB OD-WeaponDetection zip to Drive once, then converts it. If Google Drive refuses the
download (quota), download `OD-WeaponDetection-master.zip` in a browser from
https://drive.google.com/file/d/1Szc920DAh5kU8Qk38Doq0znEVR1QmTZS, put it in `MyDrive/sentinel360-weapon/`,
and run this cell again.

In [ ]:
import os
if not os.path.exists('/content/weapons/data.yaml'):
    !rm -rf /content/weapons
    !python scripts/prepare_weapon_dataset.py --zip "$WORK/OD-WeaponDetection-master.zip" --out /content/weapons
assert os.path.exists('/content/weapons/data.yaml'), 'Dataset not built: see the error above, then run this cell again.'
!cat /content/weapons/data.yaml

## 2. Train (~2 h)
If the session disconnects: Run all again. The resume cell below carries on from the last epoch.

In [ ]:
last = 'runs/weapon/weapon_yolo11s/weights/last.pt'
if not GPU:
    print('No GPU: skipping training.')
elif os.path.exists(last):
    print('Previous run found: use the resume cell below instead.')
else:
    !python scripts/train_weapon.py --data /content/weapons/data.yaml --model yolo11s.pt --name weapon_yolo11s --epochs 80 --batch 32 --device 0

In [ ]:
# Resume after a disconnect (GPU only; skips itself if training already finished).
from ultralytics import YOLO
last = 'runs/weapon/weapon_yolo11s/weights/last.pt'
if GPU and os.path.exists(last):
    try:
        YOLO(last).train(resume=True)
    except Exception as e:  # already finished
        print('Nothing to resume:', e)

# Pick the run that got furthest (a resume can land in another folder) and report it
# from its results.csv, so no GPU is needed here.
import csv, glob, shutil, yaml
runs = []
for best in glob.glob('runs/**/weights/best.pt', recursive=True):
    run = os.path.dirname(os.path.dirname(best))
    rows = list(csv.DictReader(open(f'{run}/results.csv'))) if os.path.exists(f'{run}/results.csv') else []
    rows = [{k.strip(): v for k, v in r.items()} for r in rows]
    planned = yaml.safe_load(open(f'{run}/args.yaml')).get('epochs') if os.path.exists(f'{run}/args.yaml') else '?'
    top = max((float(r['metrics/mAP50(B)']) for r in rows), default=0.0)
    runs.append((len(rows), top, run, planned, best))
    print(f'{run}: {len(rows)}/{planned} epochs, best mAP50 {top:.3f}')
assert runs, 'No trained weights found yet.'
epochs, top, run, planned, best = max(runs)
print(f'\nUsing {run}: {epochs}/{planned} epochs, mAP50 {top:.3f}')
if epochs < (planned if isinstance(planned, int) else 0):
    print('*** Training did not finish. Get a GPU runtime and Run all again to continue. ***')
shutil.copy(best, 'models/weapon_yolo11s.pt')

## 3. Download the model
Copy it to `apps/ai/models/weapon_yolo11s.pt` on the laptop. **Before switching**, compare it with the current
model on your own clips (see `apps/ai/eval/README.md`):
```
uv run python scripts/evaluate.py run --set WEAPON_MODEL_PATH=models/weapon_yolo11s.pt --set 'WEAPON_ALARM_LABELS=["knife","pistol"]' --set WEAPON_EXTRA_MODEL_PATH= --set WEAPON_PERSON_MODEL_PATH=models/yolov8n.pt
```

In [ ]:
shutil.copy('models/weapon_yolo11s.pt', f'{WORK}/weapon_yolo11s.pt')
from google.colab import files
files.download('models/weapon_yolo11s.pt')